# File Handling

Python's built-in `open()` function is itself just a function that takes a
filename and a mode, and returns a file object you can read from or write to. It fits
naturally here alongside the rest of this section on functions: file I/O in Python is
almost always done *through* a small function, and understanding `open()`'s own
parameters and return value is the first step to writing one.

# Opening a file

    f = open(filename, mode)

Common modes:
- `'r'` — read (default). Errors if the file does not exist.
- `'w'` — write. Creates the file if missing, **overwrites** it if it already exists.
- `'a'` — append. Creates the file if missing, adds to the end if it exists.
- `'x'` — create. Errors if the file already exists.

# Example: reading a whole file

In [ ]:
f = open("example.txt", "r")
contents = f.read()
f.close()

print(contents)

# The problem with `open()`/`close()`

If an error happens between `open()` and `close()`, the file never gets closed.
On most systems that leaks a file handle; with write mode, it can also mean your last
few writes never get flushed to disk. The fix is the `with` statement: it calls
`close()` for you automatically, even if the code inside the block raises an
exception.

In [ ]:
with open("example.txt", "r") as f:
    contents = f.read()

# f.close() already happened here -- no need to call it yourself
print(contents)
print("File closed:", f.closed)

# Reading line by line

A file object is *iterable* — looping over it directly gives you one line at a
time, without loading the whole file into memory. This matters once files get large;
for a five-line log file it makes no practical difference, but the habit is worth
building early.

In [ ]:
with open("test.txt", "r") as f:
    for line_number, line in enumerate(f, start=1):
        print(f"line {line_number}: {line.rstrip()}")

Two other common ways to get the same lines:

In [ ]:
with open("test.txt", "r") as f:
    lines = f.readlines()          # a list of strings, one per line (keeps the '\n')

print(lines)
print(f"{len(lines)} lines")

# Writing to a file

`'w'` mode overwrites the whole file. If you only want to add new content without
touching what is already there, use `'a'` (append) instead.

In [ ]:
with open("greeting.txt", "w") as f:
    f.write("Hello from Python!\n")
    f.write("This line was written second.\n")

with open("greeting.txt", "r") as f:
    print(f.read())

In [ ]:
with open("greeting.txt", "a") as f:
    f.write("This line was appended, not overwritten.\n")

with open("greeting.txt", "r") as f:
    print(f.read())

# Wrapping file I/O in a function

This is the pattern you will use constantly: a small function that opens a file,
does something with its contents, and returns a plain Python value — the caller never
has to think about `open`/`close`/`with` at all.

In [ ]:
def count_words(filename):
    '''Return the total number of whitespace-separated words in a text file.'''
    with open(filename, "r") as f:
        text = f.read()
    return len(text.split())


def word_frequencies(filename):
    '''Return a dict mapping each lowercased word to how many times it appears.'''
    with open(filename, "r") as f:
        words = f.read().lower().split()

    counts = {}
    for word in words:
        word = word.strip(".,!?;:")
        counts[word] = counts.get(word, 0) + 1
    return counts


print("word count:", count_words("test.txt"))
print("frequencies:", word_frequencies("test.txt"))

# Cleaning up

`greeting.txt` was created by this notebook, not shipped with it -- remove it so
re-running the notebook from a clean checkout behaves the same way every time.

In [ ]:
import os

if os.path.exists("greeting.txt"):
    os.remove("greeting.txt")
    print("removed greeting.txt")

# Exercise

Write a function `line_starting_with(filename, prefix)` that returns the first
line of a file that starts with `prefix`, or `None` if no line matches. Test it on
`test.txt`.

In [ ]:
def line_starting_with(filename, prefix):
    with open(filename, "r") as f:
        for line in f:
            if line.startswith(prefix):
                return line.rstrip()
    return None


print(line_starting_with("test.txt", "Contains"))
print(line_starting_with("test.txt", "Nonexistent"))